In [ ]:
# Este codigo lee modelos metabolicos SBML y extrae todas las reacciones y sus genes asociados.
# Genera un CSV excluyendo reacciones sin GPR/genes.
# Asume GPRs solo OR (por eso min_deleciones = n_genes), PERO:
# Antes de correr, escanea los SBML y busca si existe el tag <fbc:and>.


### INSTALACION Y MONTAJE
!pip install -q "cobra==0.29.0" "python-libsbml==5.20.2"

# -q quiet
# cobra: analisis modelos metabolicos
# libsbml: manejo archivos sbml


## Conectar Colab con Drive
from google.colab import drive
drive.mount('/content/drive')


### IMPORTS Y CONFIGURACION
from cobra import io
import os, glob, pandas as pd, logging, re
from tqdm.auto import tqdm
import warnings
from datetime import datetime
import xml.etree.ElementTree as ET

# io: manejo modelos sbml (Cobra)
# os: manejo rutas y directorios
# glob: buscar archivos con patrones
# pandas: manejo de tablas como DataFrame
# logging: silenciar logs de Cobra
# re: manejo de patrones (al, chon, doc...)
# tqdm: barra progreso visual
# warnings: silenciar advertencias
# datetime: para nombrar carpetas generadas segun fecha y hora (evita sobreescribir)
# xml.etree: lectura directa del SBML como XML (chequeo fbc:and)


## Silenciar logs y warnings de Cobra:
warnings.filterwarnings('ignore')
logging.getLogger('cobra').setLevel(logging.CRITICAL)


### RUTAS
CARPETA_MODELOS = "/content/drive/MyDrive/MASH_primavera_2025/modelos_sbml"  # 211 modelos sbml
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
CARPETA_SALIDA = f"/content/drive/MyDrive/MASH_primavera_2025/out/run_{timestamp}"
os.makedirs(CARPETA_SALIDA, exist_ok=True)
SALIDA_CSV = os.path.join(CARPETA_SALIDA, "reacciones_completo.csv")

paths = sorted(glob.glob(os.path.join(CARPETA_MODELOS, "*.sbml")))
print(f"Modelos encontrados: {len(paths)}")
print(f"Carpeta salida: {CARPETA_SALIDA}\n")


### Revisar si hay "<fbc:and>" en los SBML
# Esto porque el codigo asume que no hay "OR" en los GPR

def sbml_tiene_fbc_and(xml_path: str) -> bool:
    try:
        for event, elem in ET.iterparse(xml_path, events=("start",)):
            if isinstance(elem.tag, str) and elem.tag.endswith("}and"):
                return True
        return False
    except ET.ParseError:
        # SBML mal formado como XML
        return False
    except Exception:
        return False

and_encontrados = []


for ruta in tqdm(paths, desc="Chequeando AND", unit="modelo"):
    if sbml_tiene_fbc_and(ruta):
        and_encontrados.append(ruta)

print(f"  - Modelos con <fbc:and> detectado: {len(and_encontrados)}")


### FUNCIONES AUXILIARES

# quitar posibles espacios/tabs de GPRs
def normaliza_gpr(gpr: str) -> str:
    return re.sub(r"\s+", " ", (gpr or "").strip())

# pasar rxn a string con estequiometria
def to_esteq_str(rxn) -> str:
    try:
        return rxn.reaction  # funcion COBRA
    except:
        lhs = " + ".join(f"{-c} {m.id}" for m, c in rxn.metabolites.items() if c < 0)
        rhs = " + ".join(f"{c} {m.id}" for m, c in rxn.metabolites.items() if c > 0)
        arrow = "<->" if rxn.reversibility else "->"
        return f"{lhs} {arrow} {rhs}"


### GENERAR CSV
# Columnas del CSV final (sin clasificacion de tipo GPR)
cols = [
    "modelo_id",           # ID del modelo (ej: al10, nav6)
    "sitio",               # Sitio geografico (al, nav, etc)
    "rxn_id",              # ID de la reaccion
    "rxn_nombre",          # Nombre de la reaccion
    "estequiometria",      # Ecuacion quimica
    "gpr",                 # Gene-Protein-Reaction rule (texto)
    "genes",               # Lista genes separados por coma
    "n_genes",             # Numero de genes unicos
    "min_deleciones",      # Deleciones minimas (= n_genes porque solo hay OR)
    "subsystem",           # Subsistema/pathway (PWY-XXXX)
    "ec"                   # EC number si existe
]

# Crear archivo CSV
with open(SALIDA_CSV, "w", encoding="utf-8") as f:
    f.write(",".join(cols) + "\n")

contador_total = 0
contador_sin_gpr = 0


# Procesar cada modelo
for ruta in tqdm(paths, desc="Procesando modelos", unit="modelo"):
    modelo_id = os.path.splitext(os.path.basename(ruta))[0]

    try:
        # Extraer sitio de las letras iniciales (ej: "al" de "al10")
        m_site = re.match(r'^([a-z]+)', modelo_id)
        sitio = m_site.group(1) if m_site else "desconocido"

        # Cargar modelo
        m = io.read_sbml_model(ruta)  # funcion de COBRA
        filas = []

        # Procesar cada rxn (funcion COBRA)
        for rxn in m.reactions:
            # FILTRO: eliminar rxns sin GPR real
            if not rxn.gene_reaction_rule or not rxn.genes:
                contador_sin_gpr += 1
                continue

            contador_total += 1

            # Extraer informacion de la reaccion
            gpr = normaliza_gpr(rxn.gene_reaction_rule)

            # Genes unicos segun COBRA
            genes_ids = sorted(g.id for g in rxn.genes)
            genes_str = ",".join(genes_ids)
            n_genes = len(set(genes_ids))

            # CLAVE: como solo hay OR, min_deleciones = n_genes
            min_deleciones = n_genes

            # Subsistema/pathway
            subsystem = getattr(rxn, "subsystem", "") or ""

            # EC number (buscar en anotacion o en ID)
            ec = ""
            if isinstance(rxn.annotation, dict):
                ec = rxn.annotation.get("ec-code") or rxn.annotation.get("ec_number") or ""
            if not ec:
                match = re.search(r'\d+\.\d+\.\d+\.\d+', rxn.id)
                if match:
                    ec = match.group(0)

            # Guardar fila
            filas.append({
                "modelo_id": modelo_id,
                "sitio": sitio,
                "rxn_id": rxn.id,
                "rxn_nombre": rxn.name,
                "estequiometria": to_esteq_str(rxn),
                "gpr": gpr,
                "genes": genes_str,
                "n_genes": n_genes,
                "min_deleciones": min_deleciones,
                "subsystem": subsystem,
                "ec": ec
            })

        # Escribir filas al CSV
        pd.DataFrame(filas).to_csv(
            SALIDA_CSV, mode="a", header=False, index=False, quoting=1, encoding="utf-8"
        )

    except Exception as e:
        print(f"\nError en {modelo_id}: {e}")


print(f"CSV GENERADO: {SALIDA_CSV}")
print(f"Reacciones guardadas: {contador_total:,}")
print(f"Reacciones sin GPR (eliminadas): {contador_sin_gpr:,}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelos encontrados: 211
Carpeta salida: /content/drive/MyDrive/MASH_primavera_2025/out/run_20260106_001334



Chequeando AND:   0%|          | 0/211 [00:00<?, ?modelo/s]

  - Modelos con <fbc:and> detectado: 0


Procesando modelos:   0%|          | 0/211 [00:00<?, ?modelo/s]


CSV GENERADO: /content/drive/MyDrive/MASH_primavera_2025/out/run_20260106_001334/reacciones_completo.csv
Reacciones guardadas: 231,292
Reacciones sin GPR (eliminadas): 87,868

